# Retrieval

In [1]:
# Load the environment
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Read the model name
import os
MODEL_NAME = os.environ["GEMINI_MODEL"]
API_KEY = os.environ["GOOGLE_GENERATIVE_AI_API_KEY"]

In [7]:
# Create the LangChain model
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI (model = MODEL_NAME, api_key =API_KEY, temperature = 0)

In [3]:
# Create the document object
from langchain_core.documents import Document
doc = Document(
    page_content="LangChain provides abstractions for building LLM applications.",
    metadata = {"source":"langchain_intro.md"}
)

In [4]:
print(doc)
print(doc.page_content) #.page_content contain the actual textual content
print(doc.metadata) # metadata contains additional information about the document. 

page_content='LangChain provides abstractions for building LLM applications.' metadata={'source': 'langchain_intro.md'}
LangChain provides abstractions for building LLM applications.
{'source': 'langchain_intro.md'}


## Create simple retriever

In [5]:
documents = [
    Document(
        page_content="LangChain provides abstractions for LLM applications.",
        metadata={"topic": "langchain"}
    ),
    Document(
        page_content="Python is a general-purpose programming language.",
        metadata={"topic": "python"}
    ),
    Document(
        page_content="LangGraph is used for building stateful agent workflows.",
        metadata={"topic": "langgraph"}
    ),
]


In [6]:
# Create base retriever
from langchain_core.retrievers import BaseRetriever

class TopicRetriever(BaseRetriever): # BaseRetriever gives the class the Runnable interface
    documents: list[Document]

    def _get_relevant_documents(self, query:str) -> list[Document]:

        relevant_documents = [
            doc
            for doc in self.documents
            if doc.metadata["topic"] == query
        ]
        return relevant_documents
        """
        # same as:
        relevant_documents = []

        for doc in self.documents:
            if doc.metadata["topic"] == query:
                relevant_documents.append(doc)

        return relevant_documents
        """

In [7]:
# Create and invoke retriever object

retriever = TopicRetriever(documents = documents)

results = retriever.invoke("langchain")

In [8]:
print(results)
print(type(results))
print(results[0].page_content)
print(results[0].metadata)

[Document(metadata={'topic': 'langchain'}, page_content='LangChain provides abstractions for LLM applications.')]
<class 'list'>
LangChain provides abstractions for LLM applications.
{'topic': 'langchain'}


In [9]:
# Using batch with multiple queries

# batch is used when we have multiple independent queries to retrieve at once.
batch_results = retriever.batch([
    "langchain",
    "python",
    "langgraph"
])

print(batch_results)


[[Document(metadata={'topic': 'langchain'}, page_content='LangChain provides abstractions for LLM applications.')], [Document(metadata={'topic': 'python'}, page_content='Python is a general-purpose programming language.')], [Document(metadata={'topic': 'langgraph'}, page_content='LangGraph is used for building stateful agent workflows.')]]


In [10]:
for doc in batch_results:

    print(doc)

[Document(metadata={'topic': 'langchain'}, page_content='LangChain provides abstractions for LLM applications.')]
[Document(metadata={'topic': 'python'}, page_content='Python is a general-purpose programming language.')]
[Document(metadata={'topic': 'langgraph'}, page_content='LangGraph is used for building stateful agent workflows.')]


## Retriever implementation

The architecture:

Indexing

```text
Documents
   ↓
Embedding model
   ↓
Vectors
   ↓
Vector store

```

Query time

```text
User query
   ↓
Embedding model
   ↓
Query vector
   ↓
Vector store
   ↓
Relevant Documents
   ↓
Retriever
   ↓
List[Document]
```

In [11]:
documents = [
    Document(
        page_content="LangChain provides abstractions for building LLM applications.",
        metadata={"source": "langchain.txt"}
    ),
    Document(
        page_content="LangGraph is designed for stateful agent workflows.",
        metadata={"source": "langgraph.txt"}
    ),
    Document(
        page_content="Python is a general-purpose programming language.",
        metadata={"source": "python.txt"}
    ),
]

In [ ]:
# Create the embedding model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=API_KEY,
)

In [ ]:
# Create the vector store
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embeddings) # what happened here that the documents are converted into embeddings using the embedding model then stored in the vector store.

"""
documents
    ↓
embedding model
    ↓
document embeddings
    ↓
vector store
    ├── embeddings
    └── associated Documents
"""

In [16]:
# Add documents to the vector_store

vector_store.add_documents(documents)

['09a6eb73-b017-4a0d-94d5-37fbca9c311a',
 'd8d6299b-d799-48ac-af82-c0e060af56c6',
 '504585f1-e28a-4d73-b505-cb92a54600eb']

In [20]:
# Run the retriever

retriever = vector_store.as_retriever(
    search_kwargs = {"k":2} # Top k (2 in our case) documents will be returned
)
results = retriever.invoke(
    "How do I build stateful agent workflows?"
)

print(results)

for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("---")

[Document(id='d8d6299b-d799-48ac-af82-c0e060af56c6', metadata={'source': 'langgraph.txt'}, page_content='LangGraph is designed for stateful agent workflows.'), Document(id='09a6eb73-b017-4a0d-94d5-37fbca9c311a', metadata={'source': 'langchain.txt'}, page_content='LangChain provides abstractions for building LLM applications.')]
LangGraph is designed for stateful agent workflows.
{'source': 'langgraph.txt'}
---
LangChain provides abstractions for building LLM applications.
{'source': 'langchain.txt'}
---


## Retrieving strategies 


In [ ]:
# MMR retriever
mmr_retriever = vector_store.as_retriever(
    search_type = "mmr", # MMR gives relevant documents while avoiding excessive redundancy (less redundant documents).
    search_kwargs = {
        "k": 2, # k -> how many final documents that the retriever will return.
        "fetch_k": 3, # how many candidates should MMR consider before selecting the k documents.
        #"lambda_mult": 0.6 # Controls the balance between relevance to query and diversity/less redundance. 
        # Higher emphasis on relevance means behavior closer to similarity search, while higher emphasis on diversity means stronger penalty for redundance.  
    } 
)

In [ ]:
# Similarity retriever
similarity_retriever = vector_store.as_retriever(
    search_type = "similarity", # similarity gives the most similar documents.
    search_kwargs = {"k":2} 
)

In [24]:
query = "How do I build stateful agent workflows."

In [25]:
# Invoke retrievals 
similarity_results = similarity_retriever.invoke(query)
mmr_results = mmr_retriever.invoke(query)

In [26]:
print("Similarity:")
for doc in similarity_results:
    print(doc.page_content)

print("\n MMR:")
for doc in mmr_results:
    print(doc.page_content)

Similarity:
LangGraph is designed for stateful agent workflows.
LangChain provides abstractions for building LLM applications.

 MMR:
LangGraph is designed for stateful agent workflows.
Python is a general-purpose programming language.


### Metadata

conceptual flow: 
```text
User query
    ↓
Metadata filter
    ↓
Only documents where topic == "agents"
    ↓
Semantic similarity search
    ↓
Top 2 relevant documents
    ↓
List[Document]
```

In [27]:
# Add metadata to the documents

documents = [
    Document(
        page_content="LangChain provides abstractions for building LLM applications.",
        metadata={
            "source": "langchain.txt",
            "topic": "llm"
        }
    ),
    Document(
        page_content="LangGraph is designed for stateful agent workflows.",
        metadata={
            "source": "langgraph.txt",
            "topic": "agents"
        }
    ),
    Document(
        page_content="LangGraph supports human-in-the-loop workflows.",
        metadata={
            "source": "langgraph_human.txt",
            "topic": "agents"
        }
    ),
    Document(
        page_content="Python is a general-purpose programming language.",
        metadata={
            "source": "python.txt",
            "topic": "programming"
        }
    ),
]

In [28]:
# Add documents to vector_store
vector_store.add_documents(documents)

['4e63dfd2-2736-45c4-8f6e-36e7cb0d0416',
 'c0af2c93-db2e-46da-a369-68e84f8223a7',
 '4ec183b0-86e8-4127-b31a-110b8019ef84',
 '6f7760cc-d846-4082-837f-48bc01f03f56']

In [29]:
# Create retriever with both semantic search and metadata

retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs = {
        "k": 2,
        "filter": lambda doc: doc.metadata.get("topic") == "agents"
    }
)

In [30]:
# Invoke the retriever
results = retriever.invoke(
    "How do I build stateful agent workflows?"
)

for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("---")

LangGraph is designed for stateful agent workflows.
{'source': 'langgraph.txt', 'topic': 'agents'}
---
LangGraph supports human-in-the-loop workflows.
{'source': 'langgraph_human.txt', 'topic': 'agents'}
---
